In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [6]:
documents_raw = []

for file in files:
    doc = file.parse()
    documents_raw.append(doc)

Q1

In [7]:
len(documents_raw)

72

In [8]:
from gitsource import chunk_documents

documents = chunk_documents(documents_raw, size=2000, step=1000)

Q4

In [9]:
len(documents)

295

In [10]:
from minsearch import Index

In [11]:
def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

In [12]:
index = build_index(documents)

In [13]:
query = "How does the agentic loop keep calling the model until it stops?"

In [14]:
search_result = index.search(
            query,
            num_results=5,
        )

In [15]:
search_result

[{'start': 4000,
  'content': 'while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\niteration counter so we can see how many round-trips happened.\n\n```python\nit = 1\n\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n\n        elif item.type == "message":\n            print("ASSISTANT:")\n            print(item.content[0].text)\n\n    it = it + 1\n    if has_function_calls == False:\n        break\n```\n\nThis is the core agent loop. The model rea

Q2

In [16]:
search_result[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [17]:
from rag_helper import RAGNew

In [20]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

openai_client = OpenAI()

In [33]:
assistant = RAGNew(
    index=index,
    llm_client=openai_client,
)

In [34]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [36]:
print(answer.output_text)

It keeps calling the model inside a `while True` loop. After each model response, it checks whether any item is a `function_call`:

- If there is a function call, it runs the tool, adds the result to `messages`, and keeps looping.
- If there are no function calls in that turn, it breaks out of the loop and stops.

So the stop condition is: **no function calls returned by the model in that iteration**.


In [37]:
print(answer.usage)

ResponseUsage(input_tokens=2295, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2392)


In [18]:
def calculate_usage(docs, query):
    index = build_index(docs)
    assistant = RAGNew(
    index=index,
    llm_client=openai_client,
    )
    answer = assistant.rag(query)
    print(answer.usage.input_tokens)

Q3

In [22]:
calculate_usage(documents_raw, "How does the agentic loop keep calling the model until it stops?")

7112


Q5

In [21]:
calculate_usage(documents, "How does the agentic loop keep calling the model until it stops?")

2295


Q6

In [23]:
def search(query, num_results=5):
        boost_dict = {"content": 3.0, "filename": 0.5}

        return index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
        )

In [24]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the lessons for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the content ofcourse lessons."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [25]:
import json

In [26]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [27]:
question = "How does the agentic loop work, and how is it different from plain RAG?"


In [28]:
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [29]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [30]:
it = 1
search_count = 0
while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)
    print(response.output)
    for item in response.output:
        if item.type == "function_call":
            search_count = search_count + 1
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

print(f"Search count: {search_count}")

iteration #1...
[ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG"}', call_id='call_ehM9iBNYu4tWmfsIbiSfPQMY', name='search', type='function_call', id='fc_05860fbf3058a451006a3269680b1481a2beac2415d5397e7c', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"plain RAG agentic loop"}', call_id='call_BdhQPpn2uefLog1eQ70fHYWO', name='search', type='function_call', id='fc_05860fbf3058a451006a3269680b2881a2b9560e76e861ed2f', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic workflow retrieve act reflect"}', call_id='call_oRJ684AZVq34uU8OPaL55rFE', name='search', type='function_call', id='fc_05860fbf3058a451006a3269680b3081a2adb96dec94bd2e48', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"RAG single shot retrieval answer"}', call_id='call_QP8Hs0vQUF6Jse8GEWtqF2Ln', name='search', type='function_call', id='fc_05860fbf3058a451006a3269680b3c81a2bf236a343be2b7e7', namespace